# 🧪 Test des Services AWS

Ce notebook permet de tester tous les services AWS utilisés par ReguAI :
- ☁️ Amazon S3 (stockage)
- 🧠 Amazon Comprehend (NLP)
- 🤖 Amazon Bedrock (Claude)

**Note:** Cette page a été retirée de l'application Streamlit pour garder l'interface plus simple. Utilisez ce notebook pour les tests et le debug.


## 📋 Configuration AWS


In [ ]:
import os
import json
import boto3
import pandas as pd
from datetime import datetime
from botocore.exceptions import ClientError
from pathlib import Path

# Charger .env si disponible
try:
    from dotenv import load_dotenv
    env_path = Path('../../.env')
    if env_path.exists():
        load_dotenv(env_path)
        print(f"✅ .env chargé depuis: {env_path.absolute()}")
    else:
        print(f"⚠️ .env non trouvé à: {env_path.absolute()}")
except ImportError:
    print("⚠️ python-dotenv non installé")

# Afficher la configuration
bucket_name = os.environ.get('S3_BUCKET_NAME', 'Non configuré')
region = os.environ.get('AWS_REGION', 'us-west-2')

print(f"\n📋 Configuration AWS:")
print(f"  S3 Bucket: {bucket_name}")
print(f"  Région AWS: {region}")

if bucket_name == 'Non configuré':
    print("\n❌ ERREUR: S3_BUCKET_NAME non configuré dans .env")
    print("💡 Utilisez notebooks/utils/detect_s3_from_sagemaker.ipynb pour configurer")


## ☁️ Test Amazon S3


In [ ]:
print("🔍 Test 1: Vérification accès bucket...")
try:
    s3 = boto3.client('s3', region_name=region)
    s3.head_bucket(Bucket=bucket_name)
    print(f"✅ Bucket '{bucket_name}' accessible")
except ClientError as e:
    error_code = e.response['Error']['Code']
    print(f"❌ ERREUR S3 ({error_code}): {e}")
    if error_code == '403':
        print("→ Problème de permissions IAM")
except Exception as e:
    print(f"❌ ERREUR S3: {e}")


In [ ]:
print("🔍 Test 2: Liste des objets (10 premiers)...")
try:
    response = s3.list_objects_v2(Bucket=bucket_name, MaxKeys=10)
    if 'Contents' in response:
        objects_df = pd.DataFrame([
            {'Key': obj['Key'], 'Size (bytes)': obj['Size'], 'Last Modified': obj['LastModified']}
            for obj in response['Contents']
        ])
        print(f"✅ {len(response['Contents'])} objet(s) trouvé(s)")
        display(objects_df)
    else:
        print("ℹ️ Bucket vide")
except Exception as e:
    print(f"❌ ERREUR: {e}")


In [ ]:
print("🔍 Test 3: Écriture/Lecture...")
try:
    test_key = "test_cache/test_file.json"
    test_data = {"test": "ReguAI", "timestamp": str(datetime.now())}
    
    # Écriture
    s3.put_object(
        Bucket=bucket_name,
        Key=test_key,
        Body=json.dumps(test_data).encode('utf-8'),
        ContentType='application/json'
    )
    print(f"✅ Écriture réussie: s3://{bucket_name}/{test_key}")
    
    # Lecture
    obj = s3.get_object(Bucket=bucket_name, Key=test_key)
    read_data = json.loads(obj['Body'].read().decode('utf-8'))
    print("✅ Lecture réussie:")
    print(json.dumps(read_data, indent=2))
    
    # Nettoyage
    s3.delete_object(Bucket=bucket_name, Key=test_key)
    print("🧹 Fichier test supprimé")
    
except Exception as e:
    print(f"❌ ERREUR: {e}")


## 🧠 Test Amazon Comprehend


In [ ]:
test_text = "This is a regulatory document about financial compliance requirements."
print(f"Texte à analyser: {test_text}")

try:
    comprehend = boto3.client('comprehend', region_name=region)
    
    # Test 1: Détection langue
    print("\n🔍 Test 1: Détection de langue...")
    lang_response = comprehend.detect_dominant_language(Text=test_text)
    language = lang_response['Languages'][0]['LanguageCode']
    confidence = lang_response['Languages'][0]['Score']
    print(f"✅ Langue: {language} (confiance: {confidence:.2%})")
    
    # Test 2: Extraction entités
    print("\n🔍 Test 2: Extraction d'entités...")
    entities_response = comprehend.detect_entities(Text=test_text, LanguageCode=language)
    entities = entities_response['Entities']
    if entities:
        entities_df = pd.DataFrame([
            {'Text': e['Text'], 'Type': e['Type'], 'Confidence': f"{e['Score']:.2%}"}
            for e in entities[:10]
        ])
        print(f"✅ {len(entities)} entité(s) trouvée(s)")
        display(entities_df)
    else:
        print("ℹ️ Aucune entité trouvée")
    
    # Test 3: Sentiment
    print("\n🔍 Test 3: Analyse de sentiment...")
    sentiment_response = comprehend.detect_sentiment(Text=test_text, LanguageCode=language)
    sentiment = sentiment_response['Sentiment']
    scores = sentiment_response['SentimentScore']
    print(f"✅ Sentiment: {sentiment}")
    print(f"   Confiance Positive: {scores['Positive']:.2%}")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    print(f"❌ ERREUR Comprehend ({error_code}): {e}")
    if error_code == 'AccessDeniedException':
        print("→ Problème de permissions IAM pour Comprehend")
except Exception as e:
    print(f"❌ ERREUR: {e}")


## 🤖 Test Amazon Bedrock (Claude)


In [ ]:
test_prompt = "What is the capital of France? Answer in one word."
model_id = "anthropic.claude-3-haiku-20240307-v1:0"  # Changez pour Sonnet si besoin

print(f"Prompt: {test_prompt}")
print(f"Modèle: {model_id}")

try:
    bedrock = boto3.client('bedrock-runtime', region_name=region)
    
    # Préparer requête
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "messages": [
            {
                "role": "user",
                "content": test_prompt
            }
        ]
    }
    
    print("\n🔄 Appel Bedrock en cours...")
    
    # Appel Bedrock
    response = bedrock.invoke_model(
        modelId=model_id,
        body=json.dumps(body)
    )
    
    response_body = json.loads(response['body'].read())
    answer = response_body['content'][0]['text']
    usage = response_body.get('usage', {})
    
    print("\n✅ Réponse Claude:")
    print(answer.strip())
    
    # Afficher usage
    if usage:
        print(f"\n📊 Usage:")
        print(f"  Input Tokens: {usage.get('input_tokens', 0)}")
        print(f"  Output Tokens: {usage.get('output_tokens', 0)}")
        total = usage.get('input_tokens', 0) + usage.get('output_tokens', 0)
        print(f"  Total Tokens: {total}")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    print(f"❌ ERREUR Bedrock ({error_code}): {e}")
    if error_code == 'AccessDeniedException':
        print("→ Problème de permissions IAM OU modèle non activé dans AWS Console")
        print("💡 Allez dans AWS Console > Bedrock > Model access et activez Claude")
    elif error_code == 'ModelNotReadyException':
        print("→ Modèle non prêt ou non activé")
except Exception as e:
    print(f"❌ ERREUR: {e}")


## 📊 Résumé des Tests

Vérifiez que tous les tests sont passés avant d'utiliser l'application.
